# 🩺 Mini-ChatGPT na nazwach chorób
### Sympozjum „Nowoczesne technologie w pediatrii" · Kamil Jędryczek

Ten notebook buduje **mały model językowy** — dokładnie ten sam mechanizm, na którym opiera się ChatGPT, tylko ok. 10 milionów razy mniejszy (ok. 80 tys. parametrów; modele klasy GPT-4 mają ich szacunkowo ~10¹²). Zamiast przewidywać następne **słowo** w zdaniu, przewidujemy następną **literę** w nazwie choroby.

**Jak uruchomić:** menu `Środowisko wykonawcze` → `Uruchom wszystko` (albo Ctrl+F9). Otwiera się też na telefonie; liczy w chmurze Google, nie w telefonie.

Sam trening na CPU to kilkanaście sekund (w zapisanym przebiegu ok. 15 s). W Colabie pierwsze uruchomienie, z instalacją pakietów, może potrwać kilka minut. W tym czasie można czytać komentarze 🙂

In [ ]:
# ── KONFIGURACJA ──────────────────────────────────────────────
# Który zbiór nazw trenujemy? Do wyboru: "choroby", "leki", "dinozaury"
ZBIOR = "choroby"

REPO = "https://raw.githubusercontent.com/NXTRSS/sympozjum-pediatria/main/data"
DANE = {
    "choroby":   f"{REPO}/choroby.csv",
    "leki":      f"{REPO}/leki.csv",
    "dinozaury": "https://raw.githubusercontent.com/junosuarez/dinosaurs/master/dinosaurs.csv",
}

# Nastawy dobrane automatyczną optymalizacją (eksperymenty/AUTOOPTYMALIZACJA.md):
# 12 epok z szybszym uczeniem = ok. 15 s treningu, ~98% nazw nowych.
LICZBA_EPOK = 12
TEMPO_UCZENIA = 0.003
TEMPERATURA = 0.4
ZIARNO = 42

# Ten sam kod co skrypt notebook/uruchom_demo.py. Na Colabie plik dociąga się sam.
import os, sys
from pathlib import Path
_kandydaci = [Path.cwd(), Path.cwd() / "notebook", Path.cwd().parent / "notebook"]
_katalog = next((k for k in _kandydaci if (k / "demo_model.py").exists()), None)
if _katalog is None:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/NXTRSS/sympozjum-pediatria/main/notebook/demo_model.py",
        "demo_model.py")
    _katalog = Path.cwd()
sys.path.insert(0, str(_katalog))
from demo_model import generuj, przygotuj_dane, trenuj, uruchom_aplikacje, wczytaj_nazwy, zbuduj_model

try:
    import gradio  # noqa: F401
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "gradio"], check=True)


In [ ]:
# ── DANE ──────────────────────────────────────────────────────
# Tasujemy kolejność nazw ze stałym ziarnem, więc każde uruchomienie daje tę samą kolejność.
# Ostatnie 10% po tasowaniu to sprawdzian, nie koniec alfabetu.
nazwy, zrodlo = wczytaj_nazwy(ZBIOR, ziarno=ZIARNO, dane=DANE)
print(f"Wczytano {len(nazwy)} nazw z: {zrodlo}\nPrzykłady:")
for n in nazwy[:8]:
    print("  •", n)


## Tokeny specjalne

Każdą nazwę opakowujemy w dwa znaki specjalne:
- `%` — **START**: „zacznij generować"
- `!` — **STOP**: „to już koniec nazwy"

Czyli `pneumonia` staje się `%pneumonia!`. Bez `%` model nie wiedziałby, od czego zacząć; bez `!` generowałby w nieskończoność.

In [ ]:
# ── PRZYGOTOWANIE DANYCH ──────────────────────────────────────
# Każda nazwa dostaje % na początku i ! na końcu.
# Słownik: znaki + padding (0). X to prefiks, Y to następny znak.
X, Y, znak2id, id2znak, znaki = przygotuj_dane(nazwy)
WIELKOSC_SLOWNIKA = len(znaki) + 1
print(f"Słownik ma {WIELKOSC_SLOWNIKA} tokenów: {''.join(znaki)}")
print(f"Macierz treningowa: {X.shape} (nazwy × pozycje znaków)")


## Model

Trzy warstwy, które robią całą magię:
1. **Embedding** — każda litera dostaje swój wektor liczb („znaczenie" litery),
2. **GRU** — sieć rekurencyjna z pamięcią: czyta literę po literze i pamięta kontekst,
3. **Dense + softmax** — zamienia pamięć na **rozkład prawdopodobieństwa** następnej litery.

ChatGPT różni się w środku (Transformer zamiast GRU) i skalą — ale wejście i wyjście ma identyczne: *kontekst → prawdopodobieństwa następnego tokenu*.

> **Dlaczego tylko 12 epok?** Przy dłuższym treningu model przestaje uczyć się *reguł* budowy łacińskich nazw, a zaczyna odtwarzać z pamięci konkretne nazwy ze zbioru. Sprawdziliśmy to pomiarem (katalog `eksperymenty/`): przy 80 epokach (256 jednostek, wcześniejszy zbiór 1465 nazw) tylko **10%** nazw było wystarczająco nowych — nie było ich w zbiorze ani tuż obok istniejącej nazwy. Automatyczna optymalizacja (ponad 400 pomiarów + ślepa ocena nazw) pokazała, że 12 epok z wyższym tempem uczenia daje nazwy co najmniej tak dobre jak 30 epok — w **ok. 2,5× krótszym czasie** (ok. 15 s zamiast ok. 36 s), co na żywo ma znaczenie.


In [ ]:
# ── TRENING ───────────────────────────────────────────────────
# Embedding(64) → jedna warstwa GRU(128, dropout 0.2) → softmax.
# Strata: entropia krzyżowa. Optymalizator: Adam. Ostatnie 10% nazw to sprawdzian.
model = zbuduj_model(WIELKOSC_SLOWNIKA, tempo=TEMPO_UCZENIA, ziarno=ZIARNO)
historia = trenuj(model, X, Y, epoki=LICZBA_EPOK)


In [ ]:
# ── KRZYWA UCZENIA ────────────────────────────────────────────
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(historia.history["loss"], label="trening (nazwy, które model widzi)")
plt.plot(historia.history["val_loss"], label="walidacja (nazwy odłożone na bok)")
plt.xlabel("Epoka"); plt.ylabel("Błąd (loss)")
plt.title("Dopóki walidacja spada — model uczy się reguł; gdy rośnie — zapamiętuje")
plt.legend(); plt.grid(alpha=0.3); plt.show()

## Generowanie — serce autoregresji

Pętla jest banalna:
1. pokaż modelowi dotychczasowy kontekst (zaczynamy od `%`),
2. model zwraca **rozkład prawdopodobieństwa** wszystkich możliwych następnych liter,
3. **wylosuj** literę z tego rozkładu (temperatura steruje „odwagą" losowania),
4. dopisz literę do kontekstu i wróć do punktu 1 — aż wypadnie `!`.

Model **nigdy nie mówi „nie wiem"** — zawsze zwraca jakiś rozkład i zawsze coś wylosujemy. Zapamiętajcie to zdanie przy temacie halucynacji.

Na koniec sprawdzamy rzecz kluczową: **czy ta nazwa w ogóle istnieje w zbiorze treningowym?** Jeśli nie — model niczego nie skopiował, tylko nauczył się reguł budowy łacińskich nazw i złożył z nich nową.


In [ ]:
# ── GENEROWANIE ───────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

ZNANE = set(nazwy)

def werdykt(nazwa):
    """Czy model coś wymyślił, czy tylko odtworzył nazwę ze zbioru treningowego?"""
    if nazwa in ZNANE:
        return f"⚠️  \"{nazwa}\" JUŻ ISTNIEJE w zbiorze treningowym — model ją odtworzył, nie wymyślił."
    return f"✨ \"{nazwa.upper()}\" — tej nazwy NIE MA w zbiorze treningowym. Model ją wymyślił."

def pokaz_kroki(kroki):
    """Tekstowy ślad generowania: kontekst → top 5 kandydatów → wylosowana litera."""
    for krok in kroki:
        p, w = krok["rozklad"], krok["wybor"]
        top5 = np.argsort(p)[::-1][:5]
        opis = "  ".join(f"{id2znak[i]!r}:{p[i]:.0%}" for i in top5)
        print(f"  kontekst: {krok['kontekst']:<28} → wylosowano {id2znak[w]!r:<5} "
              f"(p={p[w]:.0%})   top5: {opis}")

def histogramy(kroki, kolumny=4):
    """Rozkład prawdopodobieństwa w każdym kroku. Pomarańczowy słupek = litera wylosowana."""
    litery = [id2znak[i] for i in range(1, WIELKOSC_SLOWNIKA)]
    wiersze = int(np.ceil(len(kroki) / kolumny))
    fig, axes = plt.subplots(wiersze, kolumny, figsize=(4.6 * kolumny, 2.9 * wiersze), squeeze=False)
    for ax, krok in zip(axes.ravel(), kroki):
        p, w = krok["rozklad"][1:], krok["wybor"]
        kolory = ["#3A1C7D"] * len(p)
        kolory[w - 1] = "#FFAF7B"
        ax.bar(litery, p, color=kolory)
        ax.set_ylim(0, 1)
        ax.set_title(f"{krok['kontekst']}  →  {id2znak[w]!r}", fontsize=11)
        ax.tick_params(axis="x", labelsize=7)
        ax.grid(axis="y", alpha=0.25)
    for ax in axes.ravel()[len(kroki):]:
        ax.axis("off")
    fig.suptitle("P(następna litera | kontekst) — rozkład ostrzeje, gdy kontekst rośnie", y=1.0)
    fig.tight_layout()
    plt.show()

nazwa, kroki = generuj(model, znak2id, id2znak, temperatura=TEMPERATURA)
pierwsza_nazwa = nazwa
print("Ślad generowania — krok po kroku:\n")
pokaz_kroki(kroki)
print("\n" + werdykt(nazwa))
histogramy(kroki)


In [ ]:
# ── 10 NOWYCH NAZW ────────────────────────────────────────────
lista_nazw = []
nowe = 0
for _ in range(10):
    nazwa10, _ = generuj(model, znak2id, id2znak, temperatura=TEMPERATURA)
    lista_nazw.append(nazwa10)
    if nazwa10 in ZNANE:
        print(f" • {nazwa10:<40} ⚠️  istnieje w zbiorze treningowym")
    else:
        print(f" • {nazwa10:<40} ✨ nowa")
        nowe += 1
print(f"\n{nowe}/10 nazw nie występuje w zbiorze treningowym.")


### Eksport śladu generowania (do slajdu w prezentacji)

Poniższa komórka generuje jedną nazwę i zapisuje wszystkie kroki (rozkłady, wybory) jako JSON w formacie slajdu „Krok po kroku" (`GENERATION_STEPS`; pole `desc` uzupełnia się ręcznie). Uruchamiaj, aż wypadnie nazwa, która Ci się podoba — potem skopiuj JSON.

In [ ]:
# ── ŚLAD GENEROWANIA → JSON POD SLAJD ─────────────────────────
import json

nazwa, kroki = generuj(model, znak2id, id2znak, temperatura=TEMPERATURA)
print(werdykt(nazwa) + f"   ({len(kroki)} kroków)\n")

def pokaz(z):
    """Spacja na slajdzie byłaby niewidoczna — pokazujemy ją jako ␣."""
    return "␣" if z == " " else z

eksport = []
for krok in kroki:
    p, w = krok["rozklad"], krok["wybor"]
    top5 = np.argsort(p)[::-1][:5]
    probs = [[pokaz(id2znak[int(i)]), round(float(p[i]), 2)] for i in top5]
    if w not in top5:
        probs.append([pokaz(id2znak[w]), round(float(p[w]), 2)])
    probs.append(["…", round(max(0.0, 1 - sum(x for _, x in probs)), 2)])
    eksport.append({
        "context": "".join(pokaz(z) for z in krok["kontekst"]),
        "probs": probs,
        "chosen": pokaz(id2znak[w]),
        "chosen_p": round(float(p[w]), 2),
        "top": pokaz(id2znak[int(top5[0])]),
        "desc": "",
    })

print(json.dumps(eksport, ensure_ascii=False, indent=2))


## 📱 Opcjonalnie: własna aplikacja w Colabie

**Opcjonalnie: w Colabie uruchom, żeby dostać własną aplikację; prelegent używa skryptu `uruchom_demo.py`.**

Ta komórka odpala tę samą aplikację Gradio z publicznym linkiem (`*.gradio.live`, tymczasowym; w Gradio zwykle 72 h). Na sali prelegent jej nie uruchamia — na Macu działa tylko skrypt, żeby nie wstały dwa serwery. Publiczność z kodu QR do Colaba może ją odpalić u siebie i dostać własny link, bez logowania.


In [ ]:
# ── GRADIO ────────────────────────────────────────────────────
if os.environ.get("DEMO_BEZ_GRADIO") == "1":
    print("Pominięto Gradio (DEMO_BEZ_GRADIO=1).")
else:
    uruchom_aplikacje(model, znak2id, id2znak, ZNANE, temperatura=TEMPERATURA, zbior=ZBIOR, share=True)
